In [ ]:
import os

from rds_chat_analysis import NOTEBOOK_DIR
from rds_chat_analysis.client import init_session
from syft_core.config import CONFIG_PATH_ENV
from syft_core import Client as SyftboxClient
from syft_core.config import SyftClientConfig

In [ ]:
RDS_DO_CONFIG = "./.rds/wildchat/data_owner_config.json"
RDS_DS_CONFIG = "./.rds/wildchat/data_scientist_config.json"

DATA_OWNER_EMAIL = SyftClientConfig.load(RDS_DO_CONFIG).email

In [ ]:
# Directory we're saving all non-syftbox data to (e.g. configuration, staging folders, execution artifacts, etc.)
WORK_DIR = NOTEBOOK_DIR / "v2"

# Set env to load the correct syftbox config
os.environ[CONFIG_PATH_ENV] = RDS_DS_CONFIG

In [ ]:
ds_syftbox_client = SyftboxClient.load()

# DS connects to Data Owner's RDS server
ds_client = init_session(host=DATA_OWNER_EMAIL)

# Test if connection is working
health_check = ds_client.rpc.health()
print(f"Health check: {health_check}")
print(f"Connected to {ds_client.host} RDS server, as {ds_client.email}")
print(f"Logged in on RDS admin client: {ds_client.is_admin}")

# DS investigates available datasets

In [ ]:
wildchat_dataset = ds_client.dataset.get(name="Wildchat-postgres")
wildchat_dataset.describe()

# DS executes locally [TODO]

# DS submits a job to RDS

In [ ]:
job = ds_client.submit_job(
    vector_store_query="Messages about AI and data privacy",
    llm_query="Does this chat log contain sensitive or private information? answer with 'Yes.' or 'No.', followed by a short explanation.",
    max_vector_store_results=10,
    distance_threshold=0.4,
    filters={"language": "English"},
)

In [ ]:
job.user_code.describe()

# DS retrieves the results

Before executing the cells below, first finish the previous notebook to review, execute and share the Job as data owner.

In [ ]:
job.refresh()

job_results = ds_client.job.get_results(job)

In [ ]:
job_results.describe()

In [ ]:
job_results.outputs